# 0. Data source
Set `USE_RAW` below. With `USE_RAW = False` (default), all Part 1 cells are skipped and the analysis uses the shipped consolidated `*_full.csv`.

In [ ]:
# Data source switch
#   USE_RAW = True : run Part 1 -- merge the raw per-run simulation outputs under
#                    2_prelimit_simulation/6_ECC_saved_results_full/ (full local results, NOT shipped)
#                    and write the consolidated *_full.csv to cpp_results_formal_run/...;
#                    Part 2 then reads them from there.
#   USE_RAW = False: skip Part 1; Part 2 reads the shipped consolidated *_full.csv
#                    from 2_prelimit_simulation/4_ECC_saved_results/.
USE_RAW = False

# only consider A2, A4, B3 in this file

# 1. Merge Data

## Paths

In [ ]:
import pandas as pd
from pathlib import Path
import re
import os
import numpy as np



# search only under this directory
root = Path(
    "../../2_prelimit_simulation/6_ECC_saved_results_full/2_ECC3_benchmarks_5000p_extra"
)
model_name = "CC3"
eps_band = "Out"
results_folder_name = ("cpp_results_formal_run/2_ECC3_benchmarks_5000p_extra" if USE_RAW
                       else "../../2_prelimit_simulation/4_ECC_saved_results/2_ECC3_benchmarks_5000p_extra")
figures_folder_name = "cpp_results_formal_run/2_ECC3_benchmarks_5000p_extra"

if not os.path.exists(results_folder_name):
    os.makedirs(results_folder_name)
if not os.path.exists(figures_folder_name):
    os.makedirs(figures_folder_name)

## 1.1. A1-A4:

In [ ]:
if USE_RAW:
    for bmk in ["A2", "A4"]: #["A1", "A2", "A3", "A4"]:

        pattern = re.compile(
            fr"prelimit_simulation_results/benchmark_results/benchmark{bmk}/"
            r"rho(?P<rho>[^/]+)_safety(?P<safety>[^_]+)_epsilon(?P<epsilon>[^_]+)/pricing/shared_pooled/CC3_final_results\.csv"
        )
    
        all_dfs = []
    
        for file_path in root.rglob("CC3_final_results.csv"):
    
            path_str = str(file_path)
    
            match = pattern.search(path_str)
    
            # keep only those matching your directory structure
            if match:
                # df = pd.read_csv(file_path, header=None)
                if bmk == "A1" or bmk == "A2":
                    col_names = [
                        "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8",
                        "rho",
                        "cost","std","delay1","delay2","delay3","delay4","delay1_std","delay2_std","delay3_std","delay4_std"
                    ]
                elif bmk == "A3" or bmk == "A4":
                    col_names = [
                        "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8","param_9","param_10","param_11","param_12",
                        "rho",
                        "cost","std","delay1","delay2","delay3","delay4","delay1_std","delay2_std","delay3_std","delay4_std"
                    ]
                df = pd.read_csv(file_path, names=col_names, header=0)
    
                # print(df.shape)
                # print(df.columns)
                # print(df.head())
    
                epsilon_str = match.group("epsilon")
                df["epsilon"] = float(epsilon_str)
                # df["rho"] = float(match.group("rho"))
                # df["lambda"] = float(match.group("lambda"))
                df["safety"] = float(match.group("safety").replace("n", "-"))
    
                all_dfs.append(df)
    
        # merge
        big_df = pd.concat(all_dfs, ignore_index=True)
    
        print("Number of files read:", len(all_dfs))
        print("Total rows:", len(big_df))
    
        display(big_df.head())
    
        # save
        big_df.to_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_original.csv", index=False)

        if model_name == "RS":

            lambda1, lambda3 = 0.95*1.0/3, 0.95*1.0/3  # nominal input rate
            n = 400  # scale number between original system and heavy-traffic limit
            delay_bound1, delay_bound3 = 400.0/3, 400.0/3
            b1, b3 = delay_bound1/np.sqrt(n)*lambda1, delay_bound3/np.sqrt(n)*lambda3
            b_vec = np.array([b1, b3])
            print(f"b1 = {b1.round(2)}, b3 = {b3.round(2)}")

        elif model_name == "CC3":
            lambda_ast = 1.0/2*0.95
            n = 400  # scale number between original system and heavy-traffic limit
            inv_sqrt_n = 1 / np.sqrt(n)
            delay_bound_v, delay_bound_h = 60, 90
            b_v, b_h = delay_bound_v*lambda_ast*inv_sqrt_n, delay_bound_h*lambda_ast*inv_sqrt_n
            if bmk == "A3" or bmk == "A4" or bmk == "B3" or bmk == "B4" or bmk == "C2_abs" or bmk == "C2_rel" or bmk == "D2_abs" or bmk == "D2_rel":
                big_df['HL_rate_1'] = (big_df["param_1"]/big_df["rho"]/b_v).round(2)
                big_df['HL_rate_2'] = (big_df["param_4"]/big_df["rho"]/b_v).round(2)
                big_df['HL_rate_3'] = (big_df["param_7"]/big_df["rho"]/b_v).round(2)
                big_df['HL_rate_4'] = (big_df["param_10"]/big_df["rho"]/b_h).round(2)

        print("epsilon:", sorted(big_df["epsilon"].unique()))
        print("rho:", sorted(big_df["rho"].unique()))
        print("safety:", sorted(big_df["safety"].unique()))

        if bmk == "A1" or bmk == "A2":
            param_cols = ["safety", "rho", "epsilon", "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8"]
        elif bmk == "A3" or bmk == "A4":
            param_cols = ["safety", "rho", "epsilon", "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8","param_9","param_10","param_11","param_12"]
        
        big_df = big_df.sort_values(
            by=param_cols
        ).reset_index(drop=True)

        big_df.to_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_full.csv", index=False)
    

## 1.2. B1-B4

In [ ]:
if USE_RAW:
    for bmk in ["B3"]: #["B1", "B2", "B3", "B4"]:

        pattern = re.compile(
            fr"prelimit_simulation_results/benchmark_results/benchmark{bmk}/"
            r"rho(?P<rho>[^/]+)_safety(?P<safety>[^_]+)/pricing/shared_pooled/CC3_final_results\.csv"
        )
    
        all_dfs = []
    
        for file_path in root.rglob("CC3_final_results.csv"):
    
            path_str = str(file_path)
    
            match = pattern.search(path_str)
    
            # keep only those matching your directory structure
            if match:
                # df = pd.read_csv(file_path, header=None)
                if bmk == "B1" or bmk == "B2":
                    col_names = [
                        "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8",
                        "rho","rho_2","rho_3","rho_4",
                       "cost","std","delay1","delay2","delay3","delay4","delay1_std","delay2_std","delay3_std","delay4_std"
                    ]
                elif bmk == "B3" or bmk == "B4":
                    col_names = [
                        "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8","param_9","param_10","param_11","param_12",
                        "rho","rho_2","rho_3","rho_4",
                        "cost","std","delay1","delay2","delay3","delay4","delay1_std","delay2_std","delay3_std","delay4_std"
                    ]
                df = pd.read_csv(file_path, names=col_names, header=0)
    
                # print(df.shape)
                # print(df.columns)
                # print(df.head())
    
                # epsilon_str = match.group("epsilon")
                # df["epsilon"] = float(epsilon_str)
                # df["rho"] = float(match.group("rho"))
                # df["lambda"] = float(match.group("lambda"))
                df["safety"] = float(match.group("safety").replace("n", "-"))
    
                all_dfs.append(df)
    
        # merge
        big_df = pd.concat(all_dfs, ignore_index=True)
    
        print("Number of files read:", len(all_dfs))
        print("Total rows:", len(big_df))
    
        display(big_df.head())
    
        # save
        big_df.to_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_original.csv", index=False)

        if model_name == "RS":

            lambda1, lambda3 = 0.95*1.0/3, 0.95*1.0/3  # nominal input rate
            n = 400  # scale number between original system and heavy-traffic limit
            delay_bound1, delay_bound3 = 400.0/3, 400.0/3
            b1, b3 = delay_bound1/np.sqrt(n)*lambda1, delay_bound3/np.sqrt(n)*lambda3
            b_vec = np.array([b1, b3])
            print(f"b1 = {b1.round(2)}, b3 = {b3.round(2)}")

        elif model_name == "CC3":
            lambda_ast = 1.0/2*0.95
            n = 400  # scale number between original system and heavy-traffic limit
            inv_sqrt_n = 1 / np.sqrt(n)
            delay_bound_v, delay_bound_h = 60, 90
            b_v, b_h = delay_bound_v*lambda_ast*inv_sqrt_n, delay_bound_h*lambda_ast*inv_sqrt_n
            if bmk == "A3" or bmk == "A4" or bmk == "B3" or bmk == "B4" or bmk == "C2_abs" or bmk == "C2_rel" or bmk == "D2_abs" or bmk == "D2_rel":
                big_df['HL_rate_1'] = (big_df["param_1"]/big_df["rho"]/b_v).round(2)
                big_df['HL_rate_2'] = (big_df["param_4"]/big_df["rho"]/b_v).round(2)
                big_df['HL_rate_3'] = (big_df["param_7"]/big_df["rho"]/b_v).round(2)
                big_df['HL_rate_4'] = (big_df["param_10"]/big_df["rho"]/b_h).round(2)

        # print("epsilon:", sorted(big_df["epsilon"].unique()))
        print("rho:", sorted(big_df["rho"].unique()))
        print("safety:", sorted(big_df["safety"].unique()))

        if bmk == "B1" or bmk == "B2":
            param_cols = ["safety", "rho", "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8"]
        elif bmk == "B3" or bmk == "B4":
            param_cols = ["safety", "rho", "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8","param_9","param_10","param_11","param_12"]
        big_df = big_df.sort_values(
            by=param_cols
        ).reset_index(drop=True)

        big_df.to_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_full.csv", index=False)
    

# 2. Check Optimal Row

## Function: 

In [ ]:
def optimal_check(df):
    # 1. optimal parameter row

    print("Total parameter combinations:")
    print(df.shape)
    
    df_valid = df[(df['delay1'] <= 1) & (df['delay2'] <= 1) & (df['delay3'] <= 1) & (df['delay4'] <= 1)]
    
    print("Feasible solutions (delay < 1%):")
    print(df_valid.shape)
    print("\n")
    
    best_row = df_valid.loc[df_valid['cost'].idxmin()]

    best_row_copy = best_row.copy()
    best_row_copy[["cost", "std", "delay1", "delay2", "delay3", "delay4", "delay1_std", "delay2_std", "delay3_std", "delay4_std"]] = best_row_copy[["cost", "std", "delay1", "delay2", "delay3", "delay4", "delay1_std", "delay2_std", "delay3_std", "delay4_std"]].round(2)
    print(",".join(best_row_copy[show_names].astype(str).values))
    
    print("Optimal row:")
    display(best_row[show_names].to_frame().T)
    print("------------------------------------------------------------------------------")
    print(best_row[show_names])
    print("------------------------------------------------------------------------------")

    return best_row, df["rho"].unique()

In [ ]:
import pandas as pd

df_benchmarks = pd.DataFrame(columns=["rho", "rho_min", "rho_max", "rho_grid", "cost", "std", 
                                      "delay1", "delay2","delay3","delay4"])

## 2.1. A1-A4:

In [ ]:
import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re
import os


for bmk in ['A2', 'A4']: #['A1', 'A2', 'A3', 'A4']:

    if bmk == "A1" or bmk == "A2":
        show_names = [
                "safety", "rho", "epsilon",
                "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8",
                "cost","std",
                "delay1","delay2","delay3","delay4"
            ]
        param_cols = ["safety", "rho", "epsilon", 
                      "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8"
                     ]
    elif bmk == "A3" or bmk == "A4":
        show_names = [
                "safety", "rho", "epsilon",
                "HL_rate_1","param_2","param_3","HL_rate_2","param_5","param_6","HL_rate_3","param_8","param_9","HL_rate_4","param_11","param_12",
                "cost","std",
                "delay1","delay2","delay3","delay4"
            ]
        param_cols = ["safety", "rho", "epsilon", 
                      "HL_rate_1","param_2","param_3","HL_rate_2","param_5","param_6","HL_rate_3","param_8","param_9","HL_rate_4","param_11","param_12"
                     ]
    
    # Outward
    eps_band = "Out"
    
    df = pd.read_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_full.csv")
    
    df = df.sort_values(param_cols).reset_index(drop=True)
    
    # display(df)
    print(f'\n\nbmk = {bmk}')
    best_row, rho_list = optimal_check(df)

    df_benchmarks.loc[bmk] = [best_row["rho"], rho_list[0], rho_list[-1], (rho_list[1:] - rho_list[:-1]).min().round(2),  
                              best_row["cost"], best_row["std"],
                              best_row["delay1"], best_row["delay2"], best_row["delay3"], best_row["delay4"],
                              ]


## 2.2. B1-B4

In [ ]:
import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re
import os


for bmk in ['B3']: #['B1', 'B2', 'B3', 'B4']:

    if bmk == "B1" or bmk == "B2":
        show_names = [
                "safety", "rho", 
                "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8",
                "cost","std","delay1","delay2","delay3","delay4"
            ]
        param_cols = ["safety", "rho", 
                      "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8",
                     ]
    elif bmk == "B3" or bmk == "B4":
        show_names = [
                "safety", "rho", 
                "HL_rate_1","param_2","param_3","HL_rate_2","param_5","param_6","HL_rate_3","param_8","param_9","HL_rate_4","param_11","param_12",
                "cost","std","delay1","delay2","delay3","delay4"
            ]
        param_cols = ["safety", "rho", 
                      "HL_rate_1","param_2","param_3","HL_rate_2","param_5","param_6","HL_rate_3","param_8","param_9","HL_rate_4","param_11","param_12"
                      ]
    
    # Outward
    eps_band = "Out"
    
    df = pd.read_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_full.csv")
    
    df = df.sort_values(param_cols).reset_index(drop=True)
    
    # display(df)
    print(f'\n\nbmk = {bmk}')
    best_row, rho_list = optimal_check(df)

    df_benchmarks.loc[bmk] = [best_row["rho"], rho_list[0], rho_list[-1], (rho_list[1:] - rho_list[:-1]).min().round(2),  
                              best_row["cost"], best_row["std"],
                              best_row["delay1"], best_row["delay2"], best_row["delay3"], best_row["delay4"]
                             ]


# results summary

In [ ]:
display(df_benchmarks)

# 3. Best row by group

## Function:

In [ ]:
def best_by_group(
    df,
    group_col,
    cost_col="cost",
    delay1_col="delay1",
    delay2_col="delay2",
    delay3_col="delay3",
    delay4_col="delay4",
    feasibility_threshold=1,
    sort_groups=True
):
    """
    For each group in group_col:
      1) find the row with minimum cost without constraints
      2) find the row with minimum cost among feasible points
         (delay1 <= threshold and delay2 <= threshold)

    Parameters
    ----------
    df : pd.DataFrame
    group_col : str
        Column to group by, e.g. "rho", "epsilon", "lambda", "safety"
    cost_col : str
        Cost column name
    delay1_col, delay2_col : str
        Delay column names
    feasibility_threshold : float
        Feasibility threshold for both delays
    sort_groups : bool
        Whether to sort output by group_col

    Returns
    -------
    best_unconstrained : pd.DataFrame
    best_constrained : pd.DataFrame
    missing_groups : list
        Groups with no feasible solution
    """

    import pandas as pd

    required_cols = [group_col, cost_col, delay1_col, delay2_col]
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in df: {missing_cols}")

    # group values
    group_values = df[group_col].dropna().unique().tolist()
    if sort_groups:
        try:
            group_values = sorted(group_values)
        except Exception:
            pass

    # 1) unconstrained best
    idx_unconstrained = df.groupby(group_col)[cost_col].idxmin()
    best_unconstrained = df.loc[idx_unconstrained].copy()

    if sort_groups:
        best_unconstrained = best_unconstrained.sort_values(group_col)

    best_unconstrained = best_unconstrained.reset_index(drop=True)

    # 2) constrained best
    df_valid = df[
        (df[delay1_col] <= feasibility_threshold) &
        (df[delay2_col] <= feasibility_threshold) &
        (df[delay3_col] <= feasibility_threshold) &
        (df[delay4_col] <= feasibility_threshold)
    ].copy()

    best_constrained_list = []
    missing_groups = []

    for g in group_values:
        df_g_valid = df_valid[df_valid[group_col] == g]

        if len(df_g_valid) == 0:
            missing_groups.append(g)
        else:
            best_constrained_list.append(
                df_g_valid.loc[df_g_valid[cost_col].idxmin()]
            )

    if len(best_constrained_list) > 0:
        best_constrained = pd.DataFrame(best_constrained_list)
        if sort_groups:
            best_constrained = best_constrained.sort_values(group_col)
        best_constrained = best_constrained.reset_index(drop=True)
    else:
        best_constrained = pd.DataFrame(columns=df.columns)

    # print(f"Best row by {group_col} (without delay constraints):")
    # display(best_unconstrained)
    
    print(f"Best row by {group_col} (with delay constraints):")
    display(best_constrained)
    display(best_constrained['cost'])
    
    print(f"{group_col} with no feasible solution:")
    print(missing_groups)

    return best_unconstrained, best_constrained, missing_groups

In [ ]:
import matplotlib.pyplot as plt

def plot_best_feasible_cost(best_constrained, group_col, cost_col="cost"):
    if best_constrained.empty:
        print(f"No feasible points for {group_col}")
        return
    
    best_constrained = best_constrained.sort_values(group_col)

    plt.figure(figsize=(6, 4))
    plt.plot(best_constrained[group_col].to_numpy(), best_constrained[cost_col].to_numpy(), marker="o")
    plt.xlabel(group_col)
    plt.ylabel(cost_col)
    plt.title(f"Best feasible {cost_col} vs {group_col}")
    plt.grid(True)
    plt.show()

In [ ]:
def show_best_row_by_group(df, group_col_list):
    for group_col in group_col_list: 
        print("\n")
        print(f"Group by {group_col}:")
        print("\n")
        best_unconstrained, best_constrained, missing = best_by_group(
            df,
            group_col=group_col
        )
        plot_best_feasible_cost(best_constrained, group_col)

## 3.1. A1-A4:

In [ ]:
import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re
import os


for bmk in ['A2', 'A4']: #['A1', 'A2', 'A3', 'A4']:

    if bmk == "A1" or bmk == "A2":
        show_names = [
                "safety", "rho", "epsilon",
                "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8",
                "cost","std",
                "delay1","delay2","delay3","delay4"
            ]
        param_cols = ["safety", "rho", "epsilon", 
                      "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8"
                     ]
    elif bmk == "A3" or bmk == "A4":
        show_names = [
                "safety", "rho", "epsilon",
                "HL_rate_1","param_2","param_3","HL_rate_2","param_5","param_6","HL_rate_3","param_8","param_9","HL_rate_4","param_11","param_12",
                "cost","std",
                "delay1","delay2","delay3","delay4"
            ]
        param_cols = ["safety", "rho", "epsilon", 
                      "HL_rate_1","param_2","param_3","HL_rate_2","param_5","param_6","HL_rate_3","param_8","param_9","HL_rate_4","param_11","param_12"
                     ]
    
    # Outward
    eps_band = "Out"
    
    df = pd.read_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_full.csv")
    
    df = df.sort_values(param_cols).reset_index(drop=True)
    
    # display(df)
    print(f'\n\nbmk = {bmk}')
    group_col_list = ["rho", "epsilon", "safety"]
    show_best_row_by_group(df, group_col_list)


## 3.2. B1-B4:

In [ ]:
import pandas as pd
import numpy as np
import itertools
import statsmodels.api as sm
import statsmodels.formula.api as smf

from pathlib import Path
import re
import os


for bmk in ['B3']: #['B1', 'B2', 'B3', 'B4']:

    if bmk == "B1" or bmk == "B2":
        show_names = [
                "safety", "rho", 
                "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8",
                "cost","std","delay1","delay2","delay3","delay4"
            ]
        param_cols = ["safety", "rho", 
                      "param_1","param_2","param_3","param_4","param_5","param_6","param_7","param_8",
                     ]
    elif bmk == "B3" or bmk == "B4":
        show_names = [
                "safety", "rho", 
                "HL_rate_1","param_2","param_3","HL_rate_2","param_5","param_6","HL_rate_3","param_8","param_9","HL_rate_4","param_11","param_12",
                "cost","std","delay1","delay2","delay3","delay4"
            ]
        param_cols = ["safety", "rho", 
                      "HL_rate_1","param_2","param_3","HL_rate_2","param_5","param_6","HL_rate_3","param_8","param_9","HL_rate_4","param_11","param_12"
                      ]
    
    # Outward
    eps_band = "Out"
    
    df = pd.read_csv(f"{results_folder_name}/{model_name}_{eps_band}_bmk{bmk}_full.csv")
    
    df = df.sort_values(param_cols).reset_index(drop=True)
    
    # display(df)
    print(f'\n\nbmk = {bmk}')
    group_col_list = ["rho", "safety"]
    show_best_row_by_group(df, group_col_list)
